[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/finique/praxis-reproductions/blob/main/359741-momentum-transformer.ipynb)

# Trading with the Momentum Transformer — reproduction

**Kieran Wood, Sven Giegerich, Stephen Roberts & Stefan Zohren** · Oxford-Man Institute of Quantitative Finance · [arXiv 2112.08534](https://arxiv.org/abs/2112.08534) (v3, 2022)

The paper replaces the position-sizing head of a Deep Momentum Network with an attention–LSTM hybrid — a Decoder-Only Temporal Fusion Transformer — trained by SGD directly on the Sharpe ratio. Headline (Exhibit 3, raw signal, average of the 1995–2020 test windows, 50 Pinnacle futures): **Sharpe 2.54 / return 4.01% / max drawdown 1.32%**, against its own benchmarks — LSTM Deep Momentum Network 1.7, TSMOM 1.03, Long-Only 0.51.

This notebook retrains that architecture, and its LSTM predecessor, on the roll-adjusted continuous-futures reference panel (40 contracts, all four asset classes) and re-runs the comparison out-of-sample over **2017–2026**.

**Measured: Momentum Transformer 0.13 (independent fits span −0.04 to +0.40), LSTM DMN −0.09, and the plain 12-month TSMOM rule 0.52.** The two simple rules the paper uses as benchmarks transfer at roughly half their claimed Sharpe; both deep models transfer at approximately zero. The weights shipped here are the most favourable fit we obtained (0.40) — and permuting its own positions in time beats its actual timing 56% of the time. Breakeven cost is 0.14 basis points against a market that charges 1–3.

### How this notebook reproduces a deep-learning result

Every earlier Praxis reproduction is deterministic — the same data in, the same numbers out, anywhere. Training a neural network is not: identical code, seed and data land on different weights on a different device, BLAS or framework build. Inference from fixed weights *is* portable. So:

* the **weights Praxis trained** ship alongside this notebook (`359741-weights/`), and the cells below load them and re-derive every published number to the decimal;
* the **full training path** is here too (`retrain_window`) — run it to verify the pipeline that produced those weights, and to see where a fresh retrain lands. It will not land on the same decimals, and the spread across seeds is itself one of the findings;
* everything that *is* deterministic — the feature pipeline, the batching, the benchmarks, the portfolio accounting — is recomputed from scratch and matches the engine bit-for-bit.

### Scope

Same data, same logic, our exact numbers. The full C0–C7 gate suite — deflation, PBO, crisis windows, cost ladder, post-publication decay, factor spanning — lives in the report: <https://market-graphs.com/praxis/papers/359741>

**The data.** The reference panel is built from licensed Databento market data (CME Globex + ICE US) and is not redistributed here; `REBUILD.md` in this repository specifies it exactly so you can rebuild it under your own entitlement. It is the same panel the 361154 notebook uses — `361154-futures-panel.parquet`.


In [ ]:
# Pinned to the engine's versions so the numbers reproduce.
%pip install -q numpy==2.1.2 pandas==2.2.3 torch==2.11.0 pyarrow matplotlib


In [ ]:
"""Praxis reproduction — paper 359741 (arXiv 2112.08534), "Trading with the
Momentum Transformer: An Intelligent and Interpretable Architecture"
(Wood, Giegerich, Roberts & Zohren, Oxford-Man Institute, 2022).

Self-contained port of the Praxis engine blocks (dmn_features, the PyTorch
Decoder-Only TFT + LSTM Deep Momentum Network, the exemplar accounting).

Deep-learning reproductions have a problem the earlier Praxis notebooks do not:
TRAINING is not bit-portable — the same code, seed and data land on different
weights on a different device/BLAS/torch build. INFERENCE is. So this notebook
loads the exact weights Praxis trained (published alongside it) and re-derives
every number in the report from them, to the decimal. The full training path is
here too (`retrain_window`), so you can verify the pipeline that produced them
and see for yourself where a retrain lands.
"""
import json
import math

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

PPY, VOL_TARGET = 252, 0.15
FEATURE_NAMES = ["norm_daily_return", "norm_monthly_return", "norm_quarterly_return",
                 "norm_biannual_return", "norm_annual_return",
                 "macd_8_24", "macd_16_48", "macd_32_96"]
RET_DELTAS = (1, 21, 63, 126, 252)
MACD_SL = ((8, 24), (16, 48), (32, 96))
BURN_IN = 400
WINDOWS = [("2016-12-31", "2019-12-31"), ("2019-12-31", "2022-12-31"),
           ("2022-12-31", "2026-06-30")]
OOS_START, PAPER_ERA_END, POSTPUB_START = "2017-01-01", "2020-12-31", "2022-01-01"
PANEL = "361154-futures-panel.parquet"      # the same roll-adjusted panel as 361154


# ----------------------------------------------------------------- data -------
def load_panel():
    # The reference panel derives from licensed Databento (CME GLBX + ICE) market
    # data and is NOT redistributed with this repository. To run: rebuild the
    # panel against your own Databento subscription — the same adapter pattern
    # Praxis ships to clients (your entitlement, your data) — see REBUILD.md, or
    # contact MarketGraphs, then place the file next to this notebook.
    try:
        return pd.read_parquet(PANEL)
    except FileNotFoundError:
        raise SystemExit(
            f"{PANEL} not found — the panel is built from licensed Databento data and is "
            "not redistributed. Rebuild it under your own Databento entitlement (REBUILD.md), "
            "or contact MarketGraphs.")


# ------------------------------------- the DMN feature pipeline (LZR 2019) -----
def _winsorise_price(srs):
    e = srs.ewm(halflife=252)
    mu, sd = e.mean(), e.std()
    return np.maximum(np.minimum(srs, mu + 5 * sd), mu - 5 * sd)


def _daily_vol(r):
    return r.ewm(span=60, min_periods=60).std().bfill()


def _macd(srs, s, l):
    hl = lambda ts: np.log(0.5) / np.log(1 - 1 / ts)
    macd = srs.ewm(halflife=hl(s)).mean() - srs.ewm(halflife=hl(l)).mean()
    q = macd / srs.rolling(63).std().bfill()
    return q / q.rolling(252).std().bfill()


def asset_features(close):
    close = close.dropna()
    srs = _winsorise_price(close)
    rets = srs / srs.shift(1) - 1.0
    vol = _daily_vol(rets)
    av = vol * np.sqrt(PPY)
    out = pd.DataFrame({"daily_returns": rets, "daily_vol": vol, "ann_vol": av})
    out["target_returns"] = (rets * VOL_TARGET / av.shift(1)).shift(-1)
    for d, name in zip(RET_DELTAS, FEATURE_NAMES[:5]):
        out[name] = (srs / srs.shift(d) - 1.0) / vol / np.sqrt(d)
    for (s, l), name in zip(MACD_SL, FEATURE_NAMES[5:]):
        out[name] = _macd(srs, s, l)
    return out.iloc[BURN_IN:].dropna()


def build_panel(daily_close):
    fr = []
    for t in daily_close.columns:
        f = asset_features(daily_close[t])
        if len(f) > 252:
            fr.append(f.assign(ticker=t))
    return pd.concat(fr).sort_index()


# --------------------------------------------- the Momentum Transformer -------
class GLU(nn.Module):
    def __init__(self, d_in, d_out, p):
        super().__init__()
        self.drop, self.lin, self.gate = nn.Dropout(p), nn.Linear(d_in, d_out), nn.Linear(d_in, d_out)

    def forward(self, x):
        x = self.drop(x)
        return self.lin(x) * torch.sigmoid(self.gate(x))


class GRN(nn.Module):
    def __init__(self, d_in, d_h, d_out=None, p=0.0, d_ctx=None):
        super().__init__()
        self.d_out = d_out or d_in
        self.skip = None if self.d_out == d_in else nn.Linear(d_in, self.d_out)
        self.fc1, self.fc2 = nn.Linear(d_in, d_h), nn.Linear(d_h, d_h)
        self.ctx = nn.Linear(d_ctx, d_h, bias=False) if d_ctx else None
        self.glu, self.norm = GLU(d_h, self.d_out, p), nn.LayerNorm(self.d_out)

    def forward(self, x, c=None):
        skip = x if self.skip is None else self.skip(x)
        h = self.fc1(x)
        if self.ctx is not None and c is not None:
            h = h + self.ctx(c)
        return self.norm(skip + self.glu(self.fc2(F.elu(h))))


class VariableSelection(nn.Module):
    def __init__(self, m, d, p, d_ctx=None):
        super().__init__()
        self.m = m
        self.select = GRN(m * d, d, m, p, d_ctx)
        self.per_var = nn.ModuleList([GRN(d, d, d, p) for _ in range(m)])

    def forward(self, emb, c=None):
        B, T, D, m = emb.shape
        flat = emb.permute(0, 1, 3, 2).reshape(B, T, m * D)
        w = torch.softmax(self.select(flat, c), -1).unsqueeze(-2)
        trans = torch.stack([self.per_var[i](emb[..., i]) for i in range(m)], -1)
        return (w * trans).sum(-1), w.squeeze(-2)


class InterpretableMultiHeadAttention(nn.Module):
    def __init__(self, n_head, d_model, p):
        super().__init__()
        self.n_head, self.d_k = n_head, d_model // n_head
        self.q = nn.ModuleList([nn.Linear(d_model, self.d_k, bias=False) for _ in range(n_head)])
        self.k = nn.ModuleList([nn.Linear(d_model, self.d_k, bias=False) for _ in range(n_head)])
        self.v = nn.Linear(d_model, self.d_k, bias=False)
        self.w_o, self.drop = nn.Linear(self.d_k, d_model, bias=False), nn.Dropout(p)

    def forward(self, x, mask):
        vs = self.v(x)
        heads, attns = [], []
        for i in range(self.n_head):
            a = (self.q[i](x) @ self.k[i](x).transpose(-2, -1)) / math.sqrt(self.d_k)
            a = torch.softmax(a.masked_fill(~mask, -1e9), -1)
            heads.append(self.drop(a @ vs)); attns.append(a)
        h = torch.stack(heads, 0).mean(0) if self.n_head > 1 else heads[0]
        return self.drop(self.w_o(h)), torch.stack(attns, 0)


class DecoderOnlyTFT(nn.Module):
    """Paper Exhibit 14 — VSN → LSTM → interpretable causal self-attention →
    TimeDistributed Dense(1, tanh): a position in (-1,1) for every timestep."""

    def __init__(self, n_features, n_static, d_model, dropout, n_head=4):
        super().__init__()
        self.embed = nn.ModuleList([nn.Linear(1, d_model) for _ in range(n_features)])
        self.static_embed = nn.Embedding(max(n_static, 1), d_model)
        self.static_vsn = GRN(d_model, d_model, d_model, dropout)
        self.ctx_vs, self.ctx_enrich = GRN(d_model, d_model, d_model, dropout), GRN(d_model, d_model, d_model, dropout)
        self.ctx_h, self.ctx_c = GRN(d_model, d_model, d_model, dropout), GRN(d_model, d_model, d_model, dropout)
        self.vsn = VariableSelection(n_features, d_model, dropout, d_ctx=d_model)
        self.lstm = nn.LSTM(d_model, d_model, batch_first=True)
        self.lstm_glu, self.lstm_norm = GLU(d_model, d_model, dropout), nn.LayerNorm(d_model)
        self.enrich = GRN(d_model, d_model, d_model, dropout, d_ctx=d_model)
        self.attn = InterpretableMultiHeadAttention(n_head, d_model, dropout)
        self.attn_glu, self.attn_norm = GLU(d_model, d_model, dropout), nn.LayerNorm(d_model)
        self.decoder = GRN(d_model, d_model, d_model, dropout)
        self.out_glu, self.out_norm = GLU(d_model, d_model, 0.0), nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, 1)

    def forward(self, x, sid):
        B, T, m = x.shape
        emb = torch.stack([self.embed[i](x[..., i:i + 1]) for i in range(m)], -1)
        s = self.static_vsn(self.static_embed(sid))
        c_vs, c_en = self.ctx_vs(s).unsqueeze(1), self.ctx_enrich(s).unsqueeze(1)
        h0, c0 = self.ctx_h(s).unsqueeze(0).contiguous(), self.ctx_c(s).unsqueeze(0).contiguous()
        sel, _ = self.vsn(emb, c_vs)
        lstm_out, _ = self.lstm(sel, (h0, c0))
        temporal = self.lstm_norm(sel + self.lstm_glu(lstm_out))
        enriched = self.enrich(temporal, c_en)
        mask = torch.ones(T, T, dtype=torch.bool, device=x.device).tril()
        att, _ = self.attn(enriched, mask)
        y = self.attn_norm(enriched + self.attn_glu(att))
        y = self.out_norm(temporal + self.out_glu(self.decoder(y)))
        return torch.tanh(self.head(y)).squeeze(-1)

    def constrain(self):
        with torch.no_grad():
            w = self.head.weight
            n = w.norm(dim=1, keepdim=True).clamp(min=1e-12)
            w.mul_(n.clamp(max=3.0) / n)


class LstmDmn(nn.Module):
    """Lim-Zohren-Roberts LSTM Deep Momentum Network (the paper's DMN benchmark)."""

    def __init__(self, n_features, n_static, d_model, dropout, n_head=0):
        super().__init__()
        self.in_drop, self.out_drop = nn.Dropout(dropout), nn.Dropout(dropout)
        self.lstm = nn.LSTM(n_features, d_model, batch_first=True)
        self.head = nn.Linear(d_model, 1)

    def forward(self, x, sid):
        h, _ = self.lstm(self.in_drop(x))
        return torch.tanh(self.head(self.out_drop(h))).squeeze(-1)

    constrain = DecoderOnlyTFT.constrain


ARCHS = {"TFT": DecoderOnlyTFT, "LSTM": LstmDmn}


def sharpe_loss(z, y, mask):
    """Paper Eq 11 — Sharpe over all asset-time pairs in the minibatch."""
    c = (z * y) * mask
    n = mask.sum().clamp(min=1.0)
    mean = c.sum() / n
    return -(mean / torch.sqrt((c.pow(2).sum() / n) - mean.pow(2) + 1e-9) * math.sqrt(PPY))


# ------------------------------------------------------------- batching -------
def _pad(g, tau):
    """The authors' `_batch_data(sliding_window=False)`: chop a contract's rows
    into non-overlapping tau-blocks from the start, zero-pad the tail, mask it."""
    X, Y = g[FEATURE_NAMES].values, g["target_returns"].values
    dates = g.index.values.astype("datetime64[ns]")
    pad = (-len(X)) % tau
    if pad:
        X = np.vstack([X, np.zeros((pad, X.shape[1]))])
        Y = np.concatenate([Y, np.zeros(pad)])
        dates = np.concatenate([dates, np.full(pad, np.datetime64("NaT"), "datetime64[ns]")])
    M = np.ones(len(Y))
    if pad:
        M[-pad:] = 0.0
    return (X.reshape(-1, tau, X.shape[1]).astype(np.float32), Y.reshape(-1, tau),
            M.reshape(-1, tau), dates.reshape(-1, tau))


def prepare_window(df, train_end, test_end, tau, ratio=0.90):
    """The authors' `ModelFeatures`: per-contract 90/10 time split of everything
    before `train_end` (the last 10% of each contract's history is validation),
    then a sliding tau-window per test day with a tau-1 buffer of training rows."""
    tv, te = df[df.index <= train_end], df[(df.index > train_end) & (df.index <= test_end)]
    tickers = sorted(t for t in tv["ticker"].unique()
                     if (tv["ticker"] == t).sum() >= tau and t in set(te["ticker"]))
    sid = {t: i for i, t in enumerate(tickers)}
    tr, va, tests = [], [], []
    for t in tickers:
        g = tv[tv["ticker"] == t]
        cut = int(ratio * len(g))
        for part, bucket in ((g.iloc[:cut], tr), (g.iloc[cut:], va)):
            if not len(part):
                continue
            X, Y, M, D = _pad(part, tau)
            bucket.append({"X": X, "Y": Y, "M": M, "DATES": D,
                           "SID": np.full(len(X), sid[t])})
        buf = pd.concat([g.iloc[-(tau - 1):], te[te["ticker"] == t]])
        n = len(buf)
        if n < tau:
            continue
        idx = np.arange(tau)[None, :] + np.arange(n - tau + 1)[:, None]
        end = idx[:, -1]
        tests.append({"X": buf[FEATURE_NAMES].values[idx].astype(np.float32),
                      "SID": np.full(len(end), sid[t]), "ticker": np.full(len(end), t),
                      "date": buf.index.values[end],
                      "target": buf["target_returns"].values[end],
                      "ann_vol": buf["ann_vol"].values[end],
                      "feat": buf[FEATURE_NAMES].values[end]})
    cat = lambda bs: {k: np.concatenate([b[k] for b in bs])
                      for k in ("X", "Y", "M", "SID", "DATES")}
    train, valid = cat(tr), cat(va)
    vd = valid["DATES"].ravel()
    uniq = np.unique(vd[~np.isnat(vd)])
    valid["DAY"] = ((np.searchsorted(uniq, vd) + 1) * (~np.isnat(vd))).reshape(valid["Y"].shape)
    valid["N_DAYS"] = len(uniq) + 1
    valid["M"] = valid["M"] * (valid["DAY"] > 0)
    for d in (train, valid):
        d["X"], d["Y"] = np.nan_to_num(d["X"]), np.nan_to_num(d["Y"])
    test = {k: np.concatenate([t[k] for t in tests])
            for k in ("X", "SID", "ticker", "date", "target", "ann_vol", "feat")}
    test["X"] = np.nan_to_num(test["X"])
    return {"train": train, "valid": valid, "test": test, "tickers": tickers,
            "n_assets": len(tickers)}


@torch.no_grad()
def predict(model, X, SID, batch=256):
    model.eval()
    return np.concatenate([model(torch.tensor(X[i:i + batch]),
                                 torch.tensor(SID[i:i + batch], dtype=torch.long)).numpy()[:, -1]
                           for i in range(0, len(X), batch)])


# ------------------------------------------------------------ accounting ------
def frame(win, pos):
    t = win["test"]
    return pd.DataFrame({"date": pd.to_datetime(t["date"]), "ticker": t["ticker"],
                         "position": pos, "returns": t["target"], "ann_vol": t["ann_vol"],
                         "captured": pos * t["target"]})


def portfolio(frames, n_assets):
    """The authors' aggregation: per window, sum captured returns across assets
    and divide by that window's asset count."""
    return pd.concat([f.dropna(subset=["captured"]).groupby("date")["captured"].sum() / n
                      for f, n in zip(frames, n_assets)]).sort_index()


def net_portfolio(frames, n_assets, bps):
    """Paper Eq 12 transaction costs."""
    out = []
    for f, n in zip(frames, n_assets):
        g = f.sort_values(["ticker", "date"]).copy()
        scaled = VOL_TARGET * g["position"] / g["ann_vol"]
        g["net"] = g["captured"] - bps * 1e-4 * scaled.groupby(g["ticker"]).diff().abs().fillna(0.0)
        out.append(g.dropna(subset=["net"]).groupby("date")["net"].sum() / n)
    return pd.concat(out).sort_index()


def bench(win, kind):
    f = win["test"]["feat"]
    if kind == "LongOnly":
        return np.ones(len(f))
    if kind == "TSMOM":
        return np.sign(f[:, FEATURE_NAMES.index("norm_annual_return")])
    y = f[:, [FEATURE_NAMES.index(c) for c in ("macd_8_24", "macd_16_48", "macd_32_96")]]
    return (y * np.exp(-(y ** 2) / 4.0) / 0.89).mean(axis=1)


def linreg(win):
    """The §3.6 simpler-proxy test: pooled OLS on the SAME eight features."""
    tr = win["train"]
    m = tr["M"].ravel() > 0
    X = tr["X"].reshape(-1, tr["X"].shape[-1])[m]
    y = tr["Y"].ravel()[m]
    coef, *_ = np.linalg.lstsq(np.column_stack([X, np.ones(len(X))]), y, rcond=None)
    return np.sign(win["test"]["feat"] @ coef[:-1] + coef[-1])


def perf(r, start=OOS_START, end=None):
    r = r.loc[start:end].dropna()
    sd = r.std(ddof=1)
    eq = (1 + r).cumprod()
    dd = float((eq / eq.cummax() - 1).min())
    return {"sharpe": round(float(r.mean() / sd * np.sqrt(PPY)), 3),
            "ann_return_%": round(float((1 + r).prod() ** (PPY / len(r)) - 1) * 100, 2),
            "ann_vol_%": round(float(sd * np.sqrt(PPY)) * 100, 2),
            "max_drawdown_%": round(abs(dd) * 100, 2),
            "hit_%": round(float((r > 0).mean()) * 100, 2), "n_days": len(r)}




# --------------------------------------------------------------- training -----
# The hyperparameters Praxis's random search selected per expanding window, from
# the paper's own Exhibit-16 grid, scored on the diversified validation Sharpe.
HP = {
      "TFT": [
            {
                  "hidden_layer_size": 160,
                  "dropout_rate": 0.4,
                  "batch_size": 64,
                  "learning_rate": 0.001,
                  "max_gradient_norm": 0.01
            },
            {
                  "hidden_layer_size": 20,
                  "dropout_rate": 0.3,
                  "batch_size": 128,
                  "learning_rate": 0.1,
                  "max_gradient_norm": 0.01
            },
            {
                  "hidden_layer_size": 80,
                  "dropout_rate": 0.3,
                  "batch_size": 32,
                  "learning_rate": 0.001,
                  "max_gradient_norm": 1.0
            }
      ],
      "LSTM": [
            {
                  "hidden_layer_size": 160,
                  "dropout_rate": 0.4,
                  "batch_size": 128,
                  "learning_rate": 0.001,
                  "max_gradient_norm": 0.01
            },
            {
                  "hidden_layer_size": 20,
                  "dropout_rate": 0.3,
                  "batch_size": 256,
                  "learning_rate": 0.1,
                  "max_gradient_norm": 0.01
            },
            {
                  "hidden_layer_size": 160,
                  "dropout_rate": 0.2,
                  "batch_size": 64,
                  "learning_rate": 0.001,
                  "max_gradient_norm": 1.0
            }
      ]
}
WEIGHTS = "359741-weights"


def _diversified_sharpe(pos, y, mask, day_idx, n_days):
    c, d, m = (pos * y * mask).reshape(-1), day_idx.reshape(-1), mask.reshape(-1)
    num = np.bincount(d, weights=c, minlength=n_days)
    den = np.bincount(d, weights=m, minlength=n_days)
    keep = den > 0
    r = num[keep] / den[keep]
    return float(r.mean() / r.std() * math.sqrt(PPY)) if len(r) >= 5 and r.std() else float("nan")


def init_keras_defaults(module):
    """Keras/TensorFlow initialisation — the authors' models are Keras and
    PyTorch's defaults differ enough to change how an LSTM trains."""
    for m in module.modules():
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.uniform_(m.weight, -0.05, 0.05)
        elif isinstance(m, nn.LSTM):
            for name, p in m.named_parameters():
                if name.startswith("weight_ih"):
                    for i in range(0, p.shape[0], m.hidden_size):
                        nn.init.xavier_uniform_(p.data[i:i + m.hidden_size])
                elif name.startswith("weight_hh"):
                    for i in range(0, p.shape[0], m.hidden_size):
                        nn.init.orthogonal_(p.data[i:i + m.hidden_size])
                elif name.startswith("bias"):
                    nn.init.zeros_(p.data)
            m.bias_ih_l0.data[m.hidden_size:2 * m.hidden_size] = 1.0     # unit forget bias


def retrain_window(arch, win, hp, seed=1000, max_epochs=300, patience=25):
    """The authors' training recipe: Sharpe loss over all asset-time pairs in the
    minibatch, Adam with gradient clipping, early stopping on the DIVERSIFIED
    validation Sharpe (mean captured return across assets per date), best weights
    restored. Validation = the last 10% of each contract's training history."""
    train, valid = win["train"], win["valid"]
    day, n_days = valid["DAY"], valid["N_DAYS"]

    torch.manual_seed(seed); np.random.seed(seed)
    model = ARCHS[arch](train["X"].shape[2], int(train["SID"].max()) + 1,
                        hp["hidden_layer_size"], hp["dropout_rate"], n_head=4)
    init_keras_defaults(model)
    opt = torch.optim.Adam(model.parameters(), lr=hp["learning_rate"])
    tX, tY, tM, tS = (torch.tensor(train["X"]), torch.tensor(train["Y"], dtype=torch.float32),
                      torch.tensor(train["M"], dtype=torch.float32),
                      torch.tensor(train["SID"], dtype=torch.long))
    vX = torch.tensor(valid["X"]); vS = torch.tensor(valid["SID"], dtype=torch.long)
    best, best_state, bad = -np.inf, None, 0
    g = torch.Generator().manual_seed(seed)
    for ep in range(max_epochs):
        model.train()
        perm = torch.randperm(len(tX), generator=g)
        for i in range(0, len(tX), hp["batch_size"]):
            j = perm[i:i + hp["batch_size"]]
            opt.zero_grad()
            loss = sharpe_loss(model(tX[j], tS[j]), tY[j], tM[j])
            if not torch.isfinite(loss):
                continue
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), hp["max_gradient_norm"])
            opt.step(); model.constrain()
        model.eval()
        with torch.no_grad():
            sr = _diversified_sharpe(model(vX, vS).numpy(), valid["Y"], valid["M"], day, n_days)
        if np.isfinite(sr) and sr > best + 1e-4:
            best, bad = sr, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break
    if best_state:
        model.load_state_dict(best_state)
    return model.eval(), best


def load_trained(arch, window):
    """Load the exact weights Praxis published for this architecture/window."""
    ck = torch.load(f"{WEIGHTS}/{arch.lower()}_w{window}.pt", map_location="cpu",
                    weights_only=False)
    m = ARCHS[arch](ck["n_features"], ck["n_static"], ck["hp"]["hidden_layer_size"],
                    ck["hp"]["dropout_rate"], n_head=4)
    m.load_state_dict(ck["state_dict"])
    return m.eval()


# ------------------------------------------------------------------ main ------
CLAIMED = {"TFT": 2.54, "LSTM": 1.70,
           "TSMOM": 1.03, "LongOnly": 0.51}


def main():
    px = load_panel()
    df = build_panel(px)
    print(f"panel: {df['ticker'].nunique()} contracts · "
          f"{df.index.min().date()}→{df.index.max().date()} · {len(df):,} contract-days\n")

    frames = {k: [] for k in ("TFT", "LSTM", "TSMOM", "MACD", "LongOnly", "LinReg")}
    n_assets = []
    for wi, (tr_end, te_end) in enumerate(WINDOWS, start=1):
        w252 = prepare_window(df, tr_end, te_end, 252)
        w63 = prepare_window(df, tr_end, te_end, 63)
        n_assets.append(w252["n_assets"])
        frames["TFT"].append(frame(w252, predict(load_trained("TFT", wi),
                                                 w252["test"]["X"], w252["test"]["SID"])))
        frames["LSTM"].append(frame(w63, predict(load_trained("LSTM", wi),
                                                 w63["test"]["X"], w63["test"]["SID"])))
        for k in ("TSMOM", "MACD", "LongOnly"):
            frames[k].append(frame(w252, bench(w252, k)))
        frames["LinReg"].append(frame(w252, linreg(w252)))
        print(f"window {wi}: train ≤{tr_end} → test {te_end} · "
              f"{w252['n_assets']} contracts · {len(w252['test']['X']):,} test rows")

    series = {k: portfolio(v, n_assets) for k, v in frames.items()}
    print(f"\nOut-of-sample {series['TFT'].index[0].date()} → "
          f"{series['TFT'].index[-1].date()} · raw signal output (paper Exhibit 3 convention)\n")
    print(f"{'strategy':32}{'Sharpe':>9}{'claimed':>9}{'return%':>9}{'vol%':>8}"
          f"{'MDD%':>8}{'hit%':>8}")
    label = {"TFT": "Momentum Transformer (TFT)", "LSTM": "LSTM Deep Momentum Network",
             "TSMOM": "Plain 12-month TSMOM", "MACD": "MACD blend",
             "LongOnly": "Long Only", "LinReg": "Linear proxy, same 8 inputs"}
    out = {}
    for k in ("TFT", "LSTM", "TSMOM", "MACD", "LongOnly", "LinReg"):
        p = perf(series[k]); out[k] = p
        c = CLAIMED.get(k)
        print(f"{label[k]:32}{p['sharpe']:>9}{('—' if c is None else c):>9}"
              f"{p['ann_return_%']:>9}{p['ann_vol_%']:>8}{p['max_drawdown_%']:>8}{p['hit_%']:>8}")

    print(f"\npost-publication only ({POSTPUB_START} →):")
    for k in ("TFT", "LSTM", "TSMOM"):
        print(f"  {label[k]:32}{perf(series[k], start=POSTPUB_START)['sharpe']:>9}")

    print("\ntransaction costs (paper Eq 12), Momentum Transformer vs the naive rule:")
    for c in (0.0, 1.0, 2.0, 3.0):
        print(f"  {c:>4.1f}bps   TFT {perf(net_portfolio(frames['TFT'], n_assets, c))['sharpe']:>7}"
              f"   TSMOM {perf(net_portfolio(frames['TSMOM'], n_assets, c))['sharpe']:>7}")
    return {"series": series, "perf": out, "frames": frames, "n_assets": n_assets}


In [ ]:
# Load the panel, rebuild the features, load the trained models, re-derive the table.
results = main()


In [ ]:
import matplotlib.pyplot as plt

eq_tft = (1 + results["series"]["TFT"]).cumprod()
eq_ts = (1 + results["series"]["TSMOM"].reindex(results["series"]["TFT"].index).fillna(0)).cumprod()
eq_ls = (1 + results["series"]["LSTM"].reindex(results["series"]["TFT"].index).fillna(0)).cumprod()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(eq_tft.index, eq_tft.values, lw=1.5,
        label=f'Momentum Transformer (measured {results["perf"]["TFT"]["sharpe"]})')
ax.plot(eq_ls.index, eq_ls.values, lw=1.2,
        label=f'LSTM Deep Momentum Network (measured {results["perf"]["LSTM"]["sharpe"]})')
ax.plot(eq_ts.index, eq_ts.values, lw=1.4,
        label=f'Plain 12-month TSMOM (measured {results["perf"]["TSMOM"]["sharpe"]})')
ax.set_title("The Momentum Transformer vs the one-line trend rule — raw signal output")
ax.set_ylabel("Growth of 1"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


### Retrain from scratch (optional)


In [ ]:
# Optional: retrain one window from scratch, exactly as the engine does — the
# same architecture, loss, early stopping and selected hyperparameters. This is
# the path that produced the shipped weights. It will NOT land on the same
# decimals (see the note at the top); the point is that you can watch it run and
# see the spread a retrain produces.
px = load_panel()
df = build_panel(px)
win = prepare_window(df, *WINDOWS[0], tau=252)
model, val_sharpe = retrain_window("TFT", win, HP["TFT"][0], seed=1000, max_epochs=60)
print(f"validation Sharpe {val_sharpe:.3f}")
pos = predict(model, win["test"]["X"], win["test"]["SID"])
print("window 1 out-of-sample:", perf(portfolio([frame(win, pos)], [win["n_assets"]]),
                                     start="2017-01-01", end="2019-12-31"))
